<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:20px 32px; border-radius:8px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

## 🪝 LangChain 的 Node 风格 Middleware

**一句话定位**：挂在 agent 循环里固定节点前后的钩子——类似 Express/Django 的 middleware，但服务的是 LLM agent 图。

</div>

### 📖 这个 notebook 讲什么

这是 node 风格 middleware 的参考教程——在 LangChain agent 循环里某些**特定节点**前后触发的钩子。

### 🧠 大图景：为什么需要 middleware

LLM agent 是一个**循环**：

```
用户消息 → LLM 思考 → （可能调用工具）→ 工具结果 → LLM 再思考 → ...
```

每一轮都在往 `messages[]` 里**追加**。轮次多了会撞上三个问题：

| 问题 | 为什么难受 |
|---|---|
| 🪙 **上下文溢出** | 即便是 128k token 的模型，长会话也会耗尽 |
| 💸 **成本爆炸** | prompt 里每个 token 每次调用都计费 |
| 🌀 **质量下降** | "lost in the middle"——prompt 一臃肿 LLM 推理就变差 |

<div class="dark-info" style="background:#1e293b; color:#e2e8f0; padding:12px 24px; border-left:4px solid #60a5fa; border-radius:4px; width:97%;"><style>.dark-info strong{color:#fbbf24;}</style>

**Middleware** 能在循环的特定节点上拦截并改写 state——在问题传到 LLM 之前解决掉。

</div>

### 🔄 agent 循环与 middleware 的挂载点

![Agent Loop](resources/agent_diagram_small_arrows_FIXED.svg)

agent 的图里有两个主要**节点**：
1. **agent 节点** —— 调用 LLM 的地方
2. **tool 节点** —— 执行工具的地方

<div class="dark-cyan" style="background:#0f2729; color:#a5f3fc; padding:10px 24px; border-left:4px solid #22d3ee; border-radius:4px; width:97%;"><style>.dark-cyan strong{color:#fbbf24;}</style>

**🆕 2026 更新**：查了官方文档和 reference.langchain.com，目前真正存在的四个 node 风格 hook 是 **`before_agent`（整次运行开始前，只跑一次）、`before_model`（每次 LLM 调用前）、`after_model`（每次 LLM 调用后）、`after_agent`（整次运行结束后，只跑一次）**。并没有独立的 `before_tools`/`after_tools` 钩子——GitHub 上有 issue（#33441）在请求补充对称的 `before_tool`/`after_tool`，但截至目前还未落地；工具层面想拦截，现在只能用 wrap 风格的 `wrap_tool_call`。下面这张"四个 hook"表格把 tool 侧的两个名字写成了 `before_tools`/`after_tools`，这是本 notebook 和实际 API 不一致的地方，记笔记时按上面纠正的名字来。

</div>

### 🪝 这四个 node 风格 hook

| Hook | 装饰器 | 触发时机 | 接收 | 返回 |
|---|---|---|---|---|
| Before Agent | `@before_agent` | 每次 LLM 调用前 | `AgentState` + `Runtime` | state 更新的 `dict`（或 `None`） |
| After Agent | `@after_agent` | LLM 响应之后 | `AgentState` + `Runtime` | state 更新的 `dict`（或 `None`） |
| Before Tools | `@before_tools` | 工具执行之前 | `AgentState` + `Runtime` | state 更新的 `dict`（或 `None`） |
| After Tools | `@after_tools` | 工具返回之后 | `AgentState` + `Runtime` | state 更新的 `dict`（或 `None`） |

In [ ]:
from dotenv import load_dotenv

load_dotenv()

---

## 📝 例 1：`SummarizationMiddleware`（内置）

### 🤔 它是什么？

一个内置的 `@before_agent` middleware（严格来说挂在 `before_model`），对话太长时自动**总结旧消息**。

原来是：
```
[msg1, msg2, msg3, msg4, msg5, msg6, msg7, msg8]  ← 一直在长
```

变成：
```
[summary_of_1_through_5, msg6, msg7, msg8]  ← 大小有界
```

agent 依然"记得"早期发生的事（通过总结），但 token 数可控。

> 🎯 可以理解成**对自己的对话历史做自动 RAG**。

### ⚙️ 参数

| 参数 | 作用 | 示例 |
|---|---|---|
| `model` | 用于总结的 LLM（可以便宜快速） | `"gpt-4o-mini"` |
| `trigger` | 什么时候触发总结 | `("tokens", 100)` 或 `("messages", 10)` |
| `keep` | 保留多少条最近消息不动 | `("messages", 1)` |

In [ ]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import SummarizationMiddleware

agent = create_agent(
    model="gpt-5-nano",
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="gpt-4o-mini",       # cheap model just for summarization
            trigger=("tokens", 100),    # summarize when messages exceed 100 tokens
            keep=("messages", 1)        # always keep the most recent 1 message
        )
    ],
)

In [ ]:
from langchain.messages import HumanMessage, AIMessage
from pprint import pprint

# Simulate a long conversation — the middleware will summarize the older messages
response = agent.invoke(
    {"messages": [
        HumanMessage(content="What is the capital of the moon?"),
        AIMessage(content="The capital of the moon is Lunapolis."),
        HumanMessage(content="What is the weather in Lunapolis?"),
        AIMessage(content="Skies are clear, with a high of 120C and a low of -100C."),
        HumanMessage(content="How many cheese miners live in Lunapolis?"),
        AIMessage(content="There are 100,000 cheese miners living in Lunapolis."),
        HumanMessage(content="Do you think the cheese miners' union will strike?"),
        AIMessage(content="Yes, because they are unhappy with the new president."),
        HumanMessage(content="If you were the new president, how would you respond?"),
    ]},
    {"configurable": {"thread_id": "1"}}
)

# The first message should now be a summary, not the original first message
print("First message (should be a summary):")
print(response["messages"][0].content[:200] + "...")

---

## ✂️ 例 2：自定义 `@before_agent` —— 裁剪 ToolMessage

### 🤔 为什么要去掉 ToolMessage？

工具结果经常包含**噪音级的内部数据**（原始 JSON、调试信息、冗长的 API 响应），LLM 没必要在后续轮次里重复看到。去掉它们能：

- 🪙 省 token
- 🎯 减少干扰
- 🔒 避免内部数据被重新泄露进 prompt

### ⚙️ `RemoveMessage` 怎么工作

LangGraph 的 `RemoveMessage` 是一种特殊消息类型，在 state 更新里返回它时，会**删除**历史记录里 `id` 匹配的那条消息。这是不直接改动 list、干净地删消息的办法。

In [ ]:
from typing import Any
from langchain.agents import AgentState
from langchain.messages import RemoveMessage, ToolMessage
from langgraph.runtime import Runtime
from langchain.agents.middleware import before_agent

@before_agent
def trim_tool_messages(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    """Remove all ToolMessages from the conversation history before each LLM call."""
    messages = state["messages"]
    tool_messages = [m for m in messages if isinstance(m, ToolMessage)]
    
    if not tool_messages:
        return None  # nothing to remove — pass through
    
    # Return RemoveMessage for each ToolMessage we want to delete
    return {"messages": [RemoveMessage(id=m.id) for m in tool_messages]}

In [ ]:
agent_trimmed = create_agent(
    model="gpt-5-nano",
    checkpointer=InMemorySaver(),
    middleware=[trim_tool_messages],
)

In [ ]:
# Conversation with noisy ToolMessages mixed in
response = agent_trimmed.invoke(
    {"messages": [
        HumanMessage(content="My device won't turn on. What should I do?"),
        ToolMessage(content="blorp-x7 initiating diagnostic ping...", tool_call_id="1"),
        AIMessage(content="Is the device plugged in and turned on?"),
        HumanMessage(content="Yes, it's plugged in and turned on."),
        ToolMessage(content="temp=42C voltage=2.9v ... greeble complete.", tool_call_id="2"),
        AIMessage(content="Is the device showing any lights or indicators?"),
        HumanMessage(content="What's the temperature of the device?")
    ]},
    {"configurable": {"thread_id": "2"}}
)

# The LLM won't see the ToolMessages — it can't answer about temperature
print(response["messages"][-1].content)

---

## 🧩 各个 Hook 该用在什么场景

| 场景 | Hook | 为什么 |
|---|---|---|
| 总结长对话 | `@before_agent`（实际应挂 `before_model`） | LLM 看之前先压缩历史 |
| 去掉噪音工具结果 | `@before_agent`（实际应挂 `before_model`） | 下次调用前清掉干扰 |
| 记录每次 LLM 响应 | `@after_agent`（实际应挂 `after_model`） | 捕获 LLM 说了什么 |
| 执行前校验工具调用 | `wrap_tool_call` | 拦截危险操作 |
| 缓存工具结果 | `wrap_tool_call` | 避免重复跑昂贵的工具 |
| 按轮次注入系统上下文 | `@before_agent`（实际应挂 `before_model`） | 给每次调用加时效性信息 |
| 从响应里脱敏 PII | `@after_agent`（实际应挂 `after_model`） | 用户看到之前清理输出 |

---

## ⚠️ 易踩坑的点

1. **middleware 的顺序有影响** —— 它们按 `middleware=[]` 里列的顺序执行。如果想同时做总结和裁剪，总结要排在裁剪前面。

2. **middleware 改的是 LLM *看到* 的东西，不是 *发生过* 的事** —— 原始消息确实存在过。middleware 只是决定把哪个版本的历史喂给 LLM。

3. **和 checkpointer 的交互** —— middleware 跑在 checkpoint 保存**之前**。所以被裁剪/总结过的版本才是真正被持久化的版本——一旦总结了，checkpoint 里就不再有原始消息了。

4. **context 是只读的** —— middleware 能读 `runtime.context`，但不能改它。只有 `state`（通过返回 dict）才能被修改。

5. **返回 `None` 表示不做改动** —— 如果这一轮 middleware 没什么要做的，返回 `None`（不是空 dict）。

---

## 🗺️ Node 风格在 Middleware 家族里的位置

| 风格 | 装饰器 | 控制什么 |
|---|---|---|
| 🪝 **Node 风格**（本 notebook） | `@before_agent`、`@after_agent`、`@before_model`、`@after_model` | 图节点边界处的 state 变换 |
| 🔀 **Wrap 风格** | `@wrap_model_call`、`@wrap_tool_call` | 真正的 LLM/工具调用（request → handler → response） |
| ⚙️ **Config 风格** | `@dynamic_prompt` | 提示词注入和 agent 配置 |
| 📦 **内置类** | `SummarizationMiddleware`、`PIIMiddleware`、`HumanInTheLoopMiddleware` 等 | 常见模式的现成方案 |

<div class="dark-success" style="background:#1a2e1f; color:#bbf7d0; padding:10px 24px; border-left:4px solid #4ade80; border-radius:4px; width:97%;"><style>.dark-success strong{color:#fbbf24;}</style>

**✨ 一句话带走**：node 风格钩子真正可用的只有 `before_agent`/`before_model`/`after_model`/`after_agent` 四个——前后两个各跑一次，中间两个每轮都跑；工具层面没有对称钩子，只能用 `wrap_tool_call`。详见 `dynamic_middleware.ipynb` 里的 wrap 风格和 config 风格。

</div>